In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Int.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Permit OrganizationalEntity": "string",
        "case:Amount": "float32",
        "case:RequestedAmount": "float32",
        "case:OriginalAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "case:AdjustedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AdjustedAmount,case:Amount,case:OriginalAmount,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,declaration 1002,2018-03-01 10:55:17,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 1002,2018-03-01 10:55:21,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,declaration 1002,2018-03-01 15:01:48,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,14787.0
3,declaration 1002,2018-03-19 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Start trip,STAFF MEMBER,EMPLOYEE,1501092.0
4,declaration 1002,2018-03-23 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,End trip,STAFF MEMBER,EMPLOYEE,345600.0
5,declaration 1002,2018-03-27 16:15:02,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,404102.0
6,declaration 1002,2018-04-03 17:07:56,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,607974.0
7,declaration 1002,2018-04-05 09:45:53,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,146277.0
8,declaration 1002,2018-04-05 17:25:23,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Request Payment,SYSTEM,UNDEFINED,27570.0
9,declaration 1002,2018-04-09 17:30:58,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Payment Handled,SYSTEM,UNDEFINED,345935.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AdjustedAmount', 'case:Amount', 'case:OriginalAmount', 'case:Permit OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [19.00, 518400.00]                       112149.0000 quantile_derived    
case:Amount                    continuous     case     yes    [28.52, 1883.08]                         375.8531   quantile_derived    
case:RequestedAmount           continuous     case    

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Permit APPROVED by SUPERVISOR', 'Permit FINAL_APPROVED by DIRECTOR'},
 {'Permit APPROVED by PRE_APPROVER', 'Permit FINAL_APPROVED by SUPERVISOR'}]

In [13]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration APPROVED by SUPERVISOR',
  'Declaration FINAL_APPROVED by DIRECTOR',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by DIRECTOR',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE',
  'End trip',
  'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by DIRECTOR',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTE

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Int-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/400 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 69274,6,1,0,0.349477,0.238953,0.460000,0.445833,0.400000,...,0.583333,0.400000,0.100000,0.200000,0.000000,0.083333,0.000000,0.000000,0.000000,0.000000
1,0,declaration 916,7,1,0,0.356235,0.262470,0.450000,0.425000,0.467647,...,0.653922,0.470588,0.100000,0.200000,0.000000,0.083333,0.000000,0.000000,0.000000,0.000000
2,0,declaration 54601,8,1,0,0.378960,0.267920,0.490000,0.475000,0.494737,...,0.432281,0.210526,0.055088,0.000000,0.110175,0.166667,0.000000,0.000000,0.000000,0.000000
3,0,declaration 30359,9,1,0,0.403310,0.216620,0.590000,0.504167,0.309524,...,0.469048,0.285714,0.100000,0.200000,0.000000,0.083333,0.000000,0.000000,0.000000,0.000000
4,0,declaration 34740,10,1,0,0.326079,0.232157,0.420000,0.379167,0.273913,...,0.784409,0.130435,0.000000,0.000000,0.000000,0.000000,0.653974,0.653974,0.999998,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,38,declaration 56837,21,1,0,0.421665,0.355236,0.488095,0.644444,0.433333,...,0.781614,0.444444,0.114948,0.047619,0.182276,0.222222,0.000000,0.000000,0.000000,0.000000
348,38,declaration 51709,21,1,0,0.400098,0.371624,0.428571,0.548611,0.433333,...,0.539015,0.444444,0.039015,0.047619,0.030411,0.055556,0.000000,0.000000,0.000000,0.000000
349,38,declaration 20507,22,1,0,0.443590,0.401467,0.485714,0.658333,0.414894,...,0.812285,0.255319,0.195855,0.095238,0.296471,0.361111,0.000000,0.825422,0.000000,1.000000
350,38,declaration 4604,23,1,0,0.408835,0.420051,0.397619,0.488889,0.261224,...,0.566702,0.244898,0.155137,0.142857,0.167417,0.166667,0.000000,0.000000,0.000000,0.000000


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 69274,6,2,0,0.512487,0.324975,0.700000,0.533333,0.333333,...,0.888889,0.333333,0.333333,0.666667,0.000000,0.222222,0.000000,0.455983,0.0,0.5
1,0,declaration 916,7,2,0,0.483869,0.284405,0.683333,0.572222,0.420000,...,0.802557,0.400000,0.180334,0.333333,0.027335,0.222222,0.000000,0.389941,0.0,0.5
2,0,declaration 54601,8,2,0,0.492541,0.318415,0.666667,0.511111,0.427273,...,1.010101,0.454545,0.333333,0.666667,0.000000,0.222222,0.000000,0.443285,0.0,0.5
3,0,declaration 30359,9,2,0,0.537018,0.357369,0.716667,0.561111,0.495833,...,1.055556,0.500000,0.333333,0.666667,0.000000,0.222222,0.000000,0.371417,0.0,0.5
4,0,declaration 34740,10,2,0,0.395690,0.191380,0.600000,0.422222,0.473077,...,0.889231,0.461538,0.205470,0.333333,0.077606,0.222222,0.000000,0.398587,0.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364,49,declaration 47409,12,5,0,0.372248,0.227829,0.516667,0.411111,0.483333,...,1.594758,0.435897,0.083333,0.000000,0.166667,0.111111,0.964416,0.964324,1.0,1.0
365,49,declaration 40053,12,5,0,0.408975,0.267950,0.550000,0.483333,0.397436,...,1.347181,0.384615,0.000000,0.000000,0.000000,0.000000,0.962566,0.962566,1.0,1.0
366,49,declaration 58488,12,5,0,0.274769,0.149539,0.400000,0.294444,0.489744,...,1.567199,0.435897,0.061099,0.000000,0.122199,0.111111,0.959091,0.962844,1.0,1.0
367,49,declaration 63108,12,5,0,0.347343,0.244686,0.450000,0.383333,0.451282,...,1.342662,0.333333,0.207672,0.333333,0.082010,0.222222,0.579435,0.969993,0.6,1.0


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()